
# Gold Sales Summary

Creates business-ready analytical Gold tables from the Silver sales dataset.

Gold tables:

- monthly_sales
- region_sales
- product_metrics
- category_metrics

These tables are designed for analytics, dashboards and the GenAI Data Analyst Copilot.

In [0]:
from pyspark.sql import functions as F

In [0]:
SILVER_TABLE = "genai_copilot.silver.sales"

MONTHLY_TABLE = "genai_copilot.gold.monthly_sales"
REGION_TABLE = "genai_copilot.gold.region_sales"
PRODUCT_TABLE = "genai_copilot.gold.product_metrics"
CATEGORY_TABLE = "genai_copilot.gold.category_metrics"

print("Source:", SILVER_TABLE)

In [0]:
silver_df = spark.table(SILVER_TABLE)

print("Silver rows:", silver_df.count())

display(silver_df.limit(10))

In [0]:
monthly_sales = (
    silver_df
    .groupBy(
        "year",
        "month"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("cost").alias("total_cost"),
        F.sum("profit").alias("total_profit"),
        F.avg("revenue").alias("average_order_value")
    )
    .withColumn(
        "year_month",
        F.concat_ws(
            "-",
            F.col("year"),
            F.lpad(F.col("month"), 2, "0")
        )
    )
    .withColumn(
        "profit_margin",
        F.when(
            F.col("total_revenue") > 0,
            F.col("total_profit") / F.col("total_revenue")
        ).otherwise(F.lit(0.0))
    )
    .select(
        "year",
        "month",
        "year_month",
        "total_orders",
        "total_quantity",
        "total_revenue",
        "total_cost",
        "total_profit",
        "profit_margin",
        "average_order_value"
    )
    .orderBy("year", "month")
)

In [0]:
display(monthly_sales)

In [0]:
(
    monthly_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(MONTHLY_TABLE)
)

print(f"Created {MONTHLY_TABLE}")

In [0]:
region_sales = (
    silver_df
    .groupBy(
        "region"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("cost").alias("total_cost"),
        F.sum("profit").alias("total_profit"),
        F.avg("revenue").alias("average_order_value")
    )
    .withColumn(
        "profit_margin",
        F.when(
            F.col("total_revenue") > 0,
            F.col("total_profit") / F.col("total_revenue")
        ).otherwise(F.lit(0.0))
    )
    .orderBy(
        F.desc("total_revenue")
    )
)

In [0]:
display(region_sales)

In [0]:
(
    region_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(REGION_TABLE)
)

print(f"Created {REGION_TABLE}")

In [0]:
product_metrics = (
    silver_df
    .groupBy(
        "product_id",
        "product_name",
        "category"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("cost").alias("total_cost"),
        F.sum("profit").alias("total_profit"),
        F.avg("revenue").alias("average_order_value")
    )
    .withColumn(
        "profit_margin",
        F.when(
            F.col("total_revenue") > 0,
            F.col("total_profit") / F.col("total_revenue")
        ).otherwise(F.lit(0.0))
    )
    .orderBy(
        F.desc("total_revenue")
    )
)

In [0]:
display(product_metrics.limit(20))

In [0]:
(
    product_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PRODUCT_TABLE)
)

print(f"Created {PRODUCT_TABLE}")

In [0]:
category_metrics = (
    silver_df
    .groupBy(
        "category"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("cost").alias("total_cost"),
        F.sum("profit").alias("total_profit"),
        F.avg("revenue").alias("average_order_value")
    )
    .withColumn(
        "profit_margin",
        F.when(
            F.col("total_revenue") > 0,
            F.col("total_profit") / F.col("total_revenue")
        ).otherwise(F.lit(0.0))
    )
    .orderBy(
        F.desc("total_revenue")
    )
)

In [0]:
display(category_metrics)

In [0]:
(
    category_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(CATEGORY_TABLE)
)

print(f"Created {CATEGORY_TABLE}")

In [0]:
print("Monthly:", spark.table(MONTHLY_TABLE).count())
print("Region:", spark.table(REGION_TABLE).count())
print("Product:", spark.table(PRODUCT_TABLE).count())
print("Category:", spark.table(CATEGORY_TABLE).count())

In [0]:
%sql
SELECT
    year_month,
    total_revenue,
    total_profit,
    profit_margin
FROM genai_copilot.gold.monthly_sales
ORDER BY year, month;

In [0]:
%sql
SELECT
    region,
    total_orders,
    total_revenue,
    total_profit,
    profit_margin
FROM genai_copilot.gold.region_sales
ORDER BY total_revenue DESC;

In [0]:
%sql
SELECT
    category,
    total_revenue,
    total_profit,
    profit_margin
FROM genai_copilot.gold.category_metrics
ORDER BY profit_margin DESC;